# Hybrid Capsule Network + Graph Neural Network for Multi-Class Leukemia Classification

**Base paper:** R. Jayakrishnan and S. Meera, *Multiclass Classification of Leukemia Cancer
Subtypes Using Gene Expression Data and Optimized Dueling Double Deep Q-Network*,
Chemometrics and Intelligent Laboratory Systems, Vol. 262, 2025, Art. 105402.

**Proposed method:** replaces the base paper's BO-CLS + Dueling Double DQN pipeline with
`SMOTE -> Capsule Network + Graph Neural Network -> Feature Fusion -> Softmax`.

This notebook runs the full pipeline (Sections 10-21 of the proposal) end to end:
Data Cleaning -> Normalization -> SMOTE -> Feature Reduction -> Capsule Network branch +
Gene Graph/GNN branch -> Feature Fusion -> Fully Connected + Softmax -> Evaluation.

Runs as-is on Google Colab / Jupyter with only `numpy, pandas, scikit-learn, networkx,
matplotlib` (all preinstalled on Colab) - no PyTorch/GPU required. A PyTorch + PyTorch
Geometric reference implementation (matching the tool stack in Proposal Section 20 exactly)
is in `torch_reference/` for GPU training on the real CuMiDa dataset.

**Using the real dataset:** download *Leukemia Gene Expression - CuMiDa* from
https://sbcb.inf.ufrgs.br/cumida , upload the CSV, and point `CSV_PATH` at it below.
Without it, this notebook uses a synthetic dataset shaped like CuMiDa (Section 4: 64
samples, thousands of genes, 5 classes) so every stage can still be demonstrated.


In [ ]:
# If running on Colab: upload/clone the project so `src/` is next to this notebook, e.g.
# !git clone <your-repo-url> && %cd <your-repo-name>/notebook
import os, sys
sys.path.insert(0, "..")   # so `src` (one level up) is importable
assert os.path.isdir("../src"), "Place this notebook inside project/notebook/ (next to ../src) or adjust the path above."


In [ ]:
import numpy as np
from src.data_utils import generate_synthetic_cumida_like, load_csv_dataset, CLASS_NAMES
from src.preprocessing import full_preprocessing_pipeline
from src.graph_construction import build_gene_graph
from src.hybrid_model import HybridCapsGNN
from src.metrics import compute_all_metrics, cross_validate_summary
import matplotlib.pyplot as plt
import networkx as nx


## 1. Gene Expression Data Acquisition (Module 1)

In [ ]:
CSV_PATH = None          # e.g. "../sample_data/cumida_leukemia.csv" once you've downloaded the real dataset
LABEL_COL = "class"

if CSV_PATH:
    X_raw, y_raw = load_csv_dataset(CSV_PATH, label_col=LABEL_COL)
    print("Loaded real dataset from", CSV_PATH)
else:
    X_raw, y_raw = generate_synthetic_cumida_like(n_samples=64, n_genes=2000, random_state=42)
    print("Using synthetic CuMiDa-like demo dataset")

print(f"{X_raw.shape[0]} samples x {X_raw.shape[1]} genes, classes: {sorted(y_raw.unique())}")
y_raw.value_counts()


## 2-3. Preprocessing & Class Balancing (Modules 2-3): Cleaning -> Normalization -> Split -> Feature Reduction -> SMOTE

In [ ]:
TOP_K_GENES = 128

prep = full_preprocessing_pipeline(X_raw, y_raw, top_k_genes=TOP_K_GENES, test_size=0.25, random_state=42)
class_names = list(prep["label_encoder"].classes_)

print("Train shape:", prep["X_train"].shape, " Test shape:", prep["X_test"].shape)
print("Class counts BEFORE SMOTE:", prep["class_counts_before_smote"])
print("Class counts AFTER  SMOTE:", prep["class_counts_after_smote"])


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
idx = list(range(len(class_names)))
before = [prep["class_counts_before_smote"].get(i,0) for i in idx]
after  = [prep["class_counts_after_smote"].get(i,0) for i in idx]
w=0.35
ax.bar([i-w/2 for i in idx], before, w, label="before SMOTE")
ax.bar([i+w/2 for i in idx], after,  w, label="after SMOTE")
ax.set_xticks(idx); ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.legend(); ax.set_title("Effect of SMOTE on class balance"); plt.show()


## 4. Graph Construction (Section 12): building the gene relationship graph G=(V,E)

In [ ]:
A_norm, G = build_gene_graph(prep["X_train"], correlation_threshold=0.5)
print(f"Gene graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

fig, ax = plt.subplots(figsize=(6,6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, node_size=40, node_color="#3D5A80", edge_color="#BBBBBB")
ax.set_title("Gene relationship graph (edges = strong expression correlation)")
plt.show()


## 5. Hybrid Capsule Network + GNN Model (Sections 11-15)

- **Capsule Network branch**: PrimaryCaps -> dynamic routing -> class capsules
  (hierarchical gene-expression features).
- **GNN branch**: 2-layer graph convolution over the gene graph -> mean-pool readout
  (relational gene-interaction features).
- **Fusion**: concatenation of both branches -> Fully Connected -> Softmax.


In [ ]:
model = HybridCapsGNN(
    n_genes=prep["X_train"].shape[1],
    n_classes=len(class_names),
    A_norm=A_norm,
    n_primary_caps=8, primary_dim=8, class_dim=12,
    gnn_hidden1=8, gnn_hidden2=8, fc_hidden=32,
    routing_iters=3, lambda_margin=0.2, seed=42,
)

EPOCHS = 30
rng = np.random.default_rng(42)
train_losses, test_accs = [], []
for epoch in range(EPOCHS):
    loss = model.train_epoch(prep["X_train"], prep["y_train"], batch_size=8, rng=rng)
    preds = model.predict(prep["X_test"])
    acc = (preds == prep["y_test"]).mean()
    train_losses.append(loss); test_accs.append(acc)
    if epoch % 5 == 0 or epoch == EPOCHS-1:
        print(f"epoch {epoch+1:>3}/{EPOCHS}  loss={loss:.4f}  test_acc={acc:.3f}")


In [ ]:
fig, ax1 = plt.subplots(figsize=(6,4))
ax1.plot(train_losses, color="tab:red"); ax1.set_ylabel("train loss", color="tab:red")
ax2 = ax1.twinx(); ax2.plot(test_accs, color="tab:blue"); ax2.set_ylabel("test accuracy", color="tab:blue")
ax1.set_xlabel("epoch"); plt.title("Training curve"); plt.show()


## 6. Evaluation (Section 19): Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix, Cross-Validation

In [ ]:
proba = model.predict_proba(prep["X_test"])
preds = np.argmax(proba, axis=1)
metrics = compute_all_metrics(prep["y_test"], preds, proba, class_names)

for k in ["accuracy","precision_macro","recall_macro","f1_macro","roc_auc_macro"]:
    print(f"{k:20s}: {metrics[k]}")


In [ ]:
cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix")
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, str(cm[i][j]), ha="center", va="center",
                color="white" if cm[i][j] > cm.max()/2 else "black")
fig.colorbar(im); plt.show()


In [ ]:
# Stratified k-fold cross-validation (recommended in Section 19 given the small sample size)
def build_and_train_fn(Xtr, ytr, Xval, yval):
    A_cv, _ = build_gene_graph(Xtr, correlation_threshold=0.5)
    m = HybridCapsGNN(n_genes=Xtr.shape[1], n_classes=len(class_names), A_norm=A_cv, seed=42)
    for _ in range(15):
        m.train_epoch(Xtr, ytr, batch_size=8)
    proba_val = m.predict_proba(Xval)
    return yval, np.argmax(proba_val, axis=1), proba_val

X_all = np.vstack([prep["X_train"], prep["X_test"]])
y_all = np.concatenate([prep["y_train"], prep["y_test"]])
cv = cross_validate_summary(build_and_train_fn, X_all, y_all, n_splits=5)
print(f"CV accuracy: {cv['mean_accuracy']:.4f} +/- {cv['std_accuracy']:.4f}")
print(f"CV F1 (macro): {cv['mean_f1_macro']:.4f} +/- {cv['std_f1_macro']:.4f}")


## 7. Single-Sample Prediction (Section 22: expected output format)

In [ ]:
sample_idx = 0
x = prep["X_test"][sample_idx]
true_label = class_names[int(prep["y_test"][sample_idx])]
probs, _ = model.forward_single(x)
pred_idx = int(np.argmax(probs))

print(f"Sample ID: S-{sample_idx:03d}")
print(f"True Class: {true_label}")
print(f"Predicted Class: {class_names[pred_idx]}")
print("Class Probabilities:")
for cn, p in sorted(zip(class_names, probs), key=lambda t: -t[1]):
    print(f"  {cn:10s}: {p:.2f}")


## 8. Existing vs Proposed (Section 16 recap)

| | Base paper (DDDQ-N) | This notebook |
|---|---|---|
| Imbalance handling | SMOTE | SMOTE |
| Feature strategy | BO-CLS | ANOVA F-score reduction + learned CapsNet/GNN representations |
| Main model | Dueling Double DQN | Capsule Network + GNN |
| Learning approach | Deep reinforcement learning | Hybrid deep representation learning |
| Gene relationships | Not explicitly modelled | Explicit graph + GNN |
| Final prediction | Q-values / actions | Softmax probabilities |

**Do not claim the proposed system outperforms the base paper without running both on the
identical real CuMiDa split** -- Section 18 of the proposal is explicit about this. Use this
notebook (or `torch_reference/train_torch.py` for a GPU run) on the real dataset before
drawing any comparative conclusion.
